In [ ]:
# --- Colab setup (version-pinned to avoid JavaPackage errors) ---
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

!pip install -q pyspark==3.5.1
!pip install -q graphframes

print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('If this is the first run after install, Runtime -> Restart runtime, then run all cells from top.')


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64
If this is the first run after install, Runtime -> Restart runtime, then run all cells from top.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from graphframes import GraphFrame

# Stop old session if any
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder     .appName('CSE4262_Assignment_GraphFrames')     .master('local[*]')     .config('spark.jars.packages', 'graphframes:graphframes:0.8.3-spark3.5-s_2.12')     .config('spark.jars.repositories', 'https://repos.spark-packages.org')     .config('spark.driver.memory', '2g')     .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark Version:', spark.version)


Spark Version: 3.5.1


In [ ]:
# -----------------------------
# Build custom collaboration network
# -----------------------------
employees = [
    ('1', 'Alice', 'Engineering'),
    ('2', 'Bob', 'Engineering'),
    ('3', 'Carol', 'Data'),
    ('4', 'David', 'Product'),
    ('5', 'Eve', 'HR'),
    ('6', 'Frank', 'Sales'),
    ('7', 'Grace', 'Engineering'),
    ('8', 'Heidi', 'Finance')
]

# (source_employee_id, target_employee_id, relationship_type)
collaborations = [
    ('1', '2', 'mentor'),
    ('2', '3', 'mentor'),
    ('3', '1', 'mentor'),   # mentor cycle
    ('4', '5', 'mentor'),
    ('6', '5', 'mentor'),
    ('7', '4', 'mentor'),
    ('8', '7', 'mentor'),
    ('1', '7', 'mentor'),
    ('5', '6', 'mentor'),

    ('1', '4', 'reports_to'),
    ('4', '8', 'reports_to'),  # chain 1->4->8
    ('2', '4', 'reports_to'),
    ('3', '4', 'reports_to'),
    ('5', '8', 'reports_to'),
    ('6', '8', 'reports_to'),
    ('7', '8', 'reports_to'),

    ('1', '2', 'colleague'),
    ('2', '1', 'colleague'),
    ('2', '3', 'colleague'),
    ('3', '2', 'colleague'),
    ('4', '7', 'colleague'),
    ('7', '4', 'colleague'),
    ('5', '8', 'colleague')
]

# (employee_id, project_id)
projects = [
    ('1', 'P1'), ('2', 'P1'), ('2', 'P2'), ('3', 'P2'),
    ('4', 'P3'), ('6', 'P3'), ('5', 'P4'), ('8', 'P5'), ('7', 'P1')
]

vertices = spark.createDataFrame(employees, ['id', 'name', 'department'])
edges = spark.createDataFrame(collaborations, ['src', 'dst', 'relationship_type'])\
    .dropDuplicates(['src', 'dst', 'relationship_type'])
projects_df = spark.createDataFrame(projects, ['employee_id', 'project_id'])

g = GraphFrame(vertices, edges)

print('Employees:')
vertices.show(truncate=False)
print('Collaborations:')
edges.show(truncate=False)
print('Projects:')
projects_df.show(truncate=False)


Employees:
+---+-----+-----------+
|id |name |department |
+---+-----+-----------+
|1  |Alice|Engineering|
|2  |Bob  |Engineering|
|3  |Carol|Data       |
|4  |David|Product    |
|5  |Eve  |HR         |
|6  |Frank|Sales      |
|7  |Grace|Engineering|
|8  |Heidi|Finance    |
+---+-----+-----------+

Collaborations:
+---+---+-----------------+
|src|dst|relationship_type|
+---+---+-----------------+
|2  |3  |mentor           |
|4  |5  |mentor           |
|1  |2  |mentor           |
|1  |7  |mentor           |
|4  |8  |reports_to       |
|3  |1  |mentor           |
|7  |4  |mentor           |
|6  |5  |mentor           |
|8  |7  |mentor           |
|1  |4  |reports_to       |
|5  |6  |mentor           |
|7  |8  |reports_to       |
|6  |8  |reports_to       |
|5  |8  |colleague        |
|2  |3  |colleague        |
|4  |7  |colleague        |
|7  |4  |colleague        |
|3  |2  |colleague        |
|3  |4  |reports_to       |
|2  |4  |reports_to       |
+---+---+-----------------+
only showing

## Task 1: Find all sets of three employees where A mentors B, B mentors C, C mentors A


In [ ]:
mentor_cycles = g.find('(a)-[ab]->(b); (b)-[bc]->(c); (c)-[ca]->(a)') \
    .filter("ab.relationship_type = 'mentor' AND bc.relationship_type = 'mentor' AND ca.relationship_type = 'mentor'") \
    .selectExpr('a.id as A_id', 'a.name as A_name', 'b.id as B_id', 'b.name as B_name', 'c.id as C_id', 'c.name as C_name')

mentor_cycles_unique = mentor_cycles.filter("CAST(A_id AS INT) < CAST(B_id AS INT) AND CAST(B_id AS INT) < CAST(C_id AS INT)")
mentor_cycles_unique.show(truncate=False)


+----+------+----+------+----+------+
|A_id|A_name|B_id|B_name|C_id|C_name|
+----+------+----+------+----+------+
|1   |Alice |2   |Bob   |3   |Carol |
+----+------+----+------+----+------+



## Task 2: Identify patterns where A reports_to B, B reports_to C


In [ ]:
report_chains = g.find('(a)-[ab]->(b); (b)-[bc]->(c)') \
    .filter("ab.relationship_type = 'reports_to' AND bc.relationship_type = 'reports_to'") \
    .selectExpr('a.id as A_id', 'a.name as A_name', 'b.id as B_id', 'b.name as B_name', 'c.id as C_id', 'c.name as C_name')

report_chains.show(truncate=False)


+----+------+----+------+----+------+
|A_id|A_name|B_id|B_name|C_id|C_name|
+----+------+----+------+----+------+
|3   |Carol |4   |David |8   |Heidi |
|1   |Alice |4   |David |8   |Heidi |
|2   |Bob   |4   |David |8   |Heidi |
+----+------+----+------+----+------+



## Task 3: Identify all pairs where A mentors B, B does NOT mentor A


In [ ]:
mentor_edges = edges.filter("relationship_type = 'mentor'")
reverse_mentor_edges = mentor_edges.selectExpr('dst as src', 'src as dst')

asymmetric_mentors = mentor_edges.alias('m') \
    .join(reverse_mentor_edges.alias('r'), on=['src', 'dst'], how='left_anti')

asymmetric_mentors_named = asymmetric_mentors \
    .join(vertices.selectExpr('id as src', 'name as src_name'), on='src') \
    .join(vertices.selectExpr('id as dst', 'name as dst_name'), on='dst') \
    .selectExpr('src as mentor_id', 'src_name as mentor_name', 'dst as mentee_id', 'dst_name as mentee_name')

asymmetric_mentors_named.show(truncate=False)


+---------+-----------+---------+-----------+
|mentor_id|mentor_name|mentee_id|mentee_name|
+---------+-----------+---------+-----------+
|2        |Bob        |3        |Carol      |
|3        |Carol      |1        |Alice      |
|7        |Grace      |4        |David      |
|1        |Alice      |2        |Bob        |
|8        |Heidi      |7        |Grace      |
|1        |Alice      |7        |Grace      |
|4        |David      |5        |Eve        |
+---------+-----------+---------+-----------+



## Task 4: Employee(s) with highest incoming mentor edges (ID, Name, total incoming mentor edges)


In [ ]:
mentor_in_counts = mentor_edges.groupBy('dst').count().withColumnRenamed('count', 'incoming_mentor_edges')
max_in = mentor_in_counts.agg(F.max('incoming_mentor_edges').alias('mx')).collect()[0]['mx']

highest_incoming_mentor = mentor_in_counts \
    .filter(F.col('incoming_mentor_edges') == max_in) \
    .join(vertices, mentor_in_counts.dst == vertices.id, 'inner') \
    .select(vertices.id.alias('employee_id'), 'name', 'department', 'incoming_mentor_edges')

highest_incoming_mentor.show(truncate=False)


+-----------+-----+-----------+---------------------+
|employee_id|name |department |incoming_mentor_edges|
+-----------+-----+-----------+---------------------+
|7          |Grace|Engineering|2                    |
|5          |Eve  |HR         |2                    |
+-----------+-----+-----------+---------------------+



### Task 4 (additional view): Highest total mentees (outgoing mentor edges)


In [ ]:
mentor_out_counts = mentor_edges.groupBy('src').count().withColumnRenamed('count', 'total_mentees')
max_out = mentor_out_counts.agg(F.max('total_mentees').alias('mx')).collect()[0]['mx']

highest_total_mentees = mentor_out_counts \
    .filter(F.col('total_mentees') == max_out) \
    .join(vertices, mentor_out_counts.src == vertices.id, 'inner') \
    .select(vertices.id.alias('employee_id'), 'name', 'department', 'total_mentees')

highest_total_mentees.show(truncate=False)


+-----------+-----+-----------+-------------+
|employee_id|name |department |total_mentees|
+-----------+-----+-----------+-------------+
|1          |Alice|Engineering|2            |
+-----------+-----+-----------+-------------+



## Task 5: Employee(s) with lowest colleague connections (in-degree + out-degree for colleague edges)


In [ ]:
colleague_edges = edges.filter("relationship_type = 'colleague'")

col_in = colleague_edges.groupBy('dst').count().withColumnRenamed('count', 'col_in')
col_out = colleague_edges.groupBy('src').count().withColumnRenamed('count', 'col_out')

col_total = vertices \
    .join(col_in, vertices.id == col_in.dst, 'left') \
    .join(col_out, vertices.id == col_out.src, 'left') \
    .select(
        vertices.id.alias('employee_id'),
        'name',
        'department',
        (F.coalesce(F.col('col_in'), F.lit(0)) + F.coalesce(F.col('col_out'), F.lit(0))).alias('colleague_connections')
    )

min_col = col_total.agg(F.min('colleague_connections').alias('mn')).collect()[0]['mn']
lowest_colleague = col_total.filter(F.col('colleague_connections') == min_col)

lowest_colleague.show(truncate=False)


+-----------+-----+----------+---------------------+
|employee_id|name |department|colleague_connections|
+-----------+-----+----------+---------------------+
|6          |Frank|Sales     |0                    |
+-----------+-----+----------+---------------------+



## Task 6: Label Propagation (LPA) communities


In [ ]:
lpa_result = g.labelPropagation(maxIter=5)

communities = lpa_result.select(
    F.col("id").alias("employee_id"),
    F.col("name"),
    F.col("department"),
    F.col("label").alias("community_id")
)

communities.orderBy("community_id", "employee_id").show(truncate=False)


+-----------+-----+-----------+-------------+
|employee_id|name |department |community_id |
+-----------+-----+-----------+-------------+
|4          |David|Product    |25769803776  |
|6          |Frank|Sales      |420906795008 |
|8          |Heidi|Finance    |420906795008 |
|5          |Eve  |HR         |644245094400 |
|2          |Bob  |Engineering|1236950581248|
|7          |Grace|Engineering|1425929142272|
|1          |Alice|Engineering|1623497637888|
|3          |Carol|Data       |1623497637888|
+-----------+-----+-----------+-------------+



## Task 7: PageRank top 5 and comparison with highest incoming mentor count


In [ ]:
pr = g.pageRank(resetProbability=0.15, maxIter=10)

top5_pr = pr.vertices.select(
    F.col("id").alias("employee_id"),
    F.col("name"),
    F.col("department"),
    F.col("pagerank")
).orderBy(F.col("pagerank").desc())

print("Top 5 employees by PageRank:")
top5_pr.show(5, truncate=False)

print("Employee(s) with highest incoming mentor edges:")
highest_incoming_mentor.show(truncate=False)

print("Short comparison:")
print("- Incoming mentor edge count measures only direct incoming mentor links.")
print("- PageRank uses the full directed network and indirect influence paths.")
print("- So rankings can differ even if someone has high incoming mentor edges.")


Top 5 employees by PageRank:
+-----------+-----+-----------+-------------------+
|employee_id|name |department |pagerank           |
+-----------+-----+-----------+-------------------+
|7          |Grace|Engineering|2.288472745434939  |
|8          |Heidi|Finance    |1.8749997811882624 |
|4          |David|Product    |1.6788478712722217 |
|5          |Eve  |HR         |0.7851224811405075 |
|2          |Bob  |Engineering|0.37364938496080435|
+-----------+-----+-----------+-------------------+
only showing top 5 rows

Employee(s) with highest incoming mentor edges:
+-----------+-----+-----------+---------------------+
|employee_id|name |department |incoming_mentor_edges|
+-----------+-----+-----------+---------------------+
|7          |Grace|Engineering|2                    |
|5          |Eve  |HR         |2                    |
+-----------+-----+-----------+---------------------+

Short comparison:
- Incoming mentor edge count measures only direct incoming mentor links.
- PageRank use